Notebook: data_quality_checks.ipynb  
Project: EcoPackAI  

This notebook validates the engineered material dataset against predefined
data quality rules before downstream modeling and recommendation.


In [ ]:
# cell 1 Environment setup
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)



In [ ]:
# cell 2 data loading
material = pd.read_csv(r"C:\Users\Ranjit\OneDrive\Desktop\AI-Powered-Sustainable-Packaging-Recommendation-System\ml\data\final\processed\material_cleaned.csv")
product = pd.read_csv(r"C:\Users\Ranjit\OneDrive\Desktop\AI-Powered-Sustainable-Packaging-Recommendation-System\ml\data\final\processed\product_cleaned.csv")

print("Material shape:", material.shape)
print("Product shape:", product.shape)



In [ ]:
material.columns

In [ ]:
#cell 3 Schema inspection & validation for material dataset
expected_material_columns = [
    "material_id",
    "material_type",
    "strength_mpa",
    "weight_capacity",
    "biodegradability_percent",
    "co2_emission_score",
    "recyclability_percent",
    "cost_per_kg",
    "industry_use_case",
    "co2_impact_index",
    "cost_efficiency_index",
    "material_suitability_score"
]

missing_cols = set(expected_material_columns) - set(material.columns)
extra_cols = set(material.columns) - set(expected_material_columns)

assert not missing_cols, f"Missing material columns: {missing_cols}"
print("Material schema validation passed")
 

In [ ]:
# cell 4 checking missing values for material dataset
nulls = material.isnull().sum()

assert nulls.sum() == 0, f"Material dataset has missing values:\n{nulls}"
print("Material missing value check passed")


In [ ]:
# cell 5 
# Engineered scores must be 0–100
assert material["co2_impact_index"].between(0, 100).all()
assert material["cost_efficiency_index"].between(0, 100).all()
assert material["material_suitability_score"].between(0, 100).all()

# Physical & cost constraints
assert (material["cost_per_kg"] > 0).all()
assert (material["strength_mpa"] > 0).all()
assert (material["weight_capacity"] > 0).all()

print("Material range & logic validation passed")


In [ ]:
# cell 6 Material Key Integrity & Duplicates
assert material["material_id"].is_unique, "Duplicate material IDs found"
assert material.duplicated().sum() == 0, "Duplicate material rows found"

print("Material integrity checks passed")


In [ ]:
# cell 7 Schema inspection & validation for product dataset
expected_product_columns = [
    "product_id",
    "product_name",
    "fragility_index",
    "shipping_type",
    "product_weight",
    "category",
]

missing_cols = set(expected_product_columns) - set(product.columns)
extra_cols = set(product.columns) - set(expected_product_columns)

assert not missing_cols, f"Missing product columns: {missing_cols}"
print("Product schema validation passed")


In [ ]:
#cell 8 checking missing values for product dataset
nulls = product.isnull().sum()

assert nulls.sum() == 0, f"Product dataset has missing values:\n{nulls}"
print("Product missing value check passed")


In [ ]:
#cell 9 Product domain validation

# Product ID must be unique
assert product["product_id"].is_unique, "Duplicate product IDs found"

# Product weight must be positive
assert (product["product_weight"] > 0).all(), "Invalid product_weight values"

# Fragility index must be within domain
assert product["fragility_index"].between(0, 10).all(), "Fragility index out of range"

# Allowed shipping types
allowed_shipping = {"Air", "Road", "Sea"}
assert product["shipping_type"].isin(allowed_shipping).all(), "Invalid shipping_type values"

print("Product domain validation passed")


In [ ]:
#cell 10 Duplicate & Integrity Check for Product

duplicate_rows = product.duplicated().sum()
assert duplicate_rows == 0, f"Duplicate product rows found: {duplicate_rows}"

print("Product integrity checks passed")


In [ ]:
#cell 11 
data_quality_status = {
    "material_dataset": {
        "rows": material.shape[0],
        "columns": material.shape[1],
        "quality_status": "PASS"
    },
    "product_dataset": {
        "rows": product.shape[0],
        "columns": product.shape[1],
        "quality_status": "PASS"
    }
}

data_quality_status
